Pada tahap preprocessing, dilakukan pembersihan dan normalisasi teks untuk meningkatkan kualitas data sebelum pemodelan. Dalam penelitian ini, digunakan dua kamus dari repository [Louis Owen](https://github.com/louisowen6/NLP_bahasa_resources), yaitu combined_root_words.txt sebagai acuan kata dasar dan combined_stop_words.txt untuk menghapus kata yang tidak relevan. Adapun tahapan yang dilakukan adalah sebagai berikut:

1.   **Case folding**: mengubah seluruh teks menjadi huruf kecil untuk konsistensi.
2. **Normalize numbers**: menyesuaikan atau menghapus angka agar tidak mengganggu analisis.
3. **Handle emoji**: mengonversi atau menghapus emoji agar tetap merepresentasikan emosi dalam teks.
4. **Remove special characters**: menghapus karakter khusus yang tidak memiliki makna linguistik.
5. **Normalize slang words**: mengubah kata tidak baku/slang menjadi bentuk baku.
6. **Tokenization**: memecah teks menjadi unit kata (token).
7. **Remove stopwords**: menghapus kata umum yang tidak memberikan informasi penting.
8. **Negation marking**: menandai kata negasi untuk menjaga konteks sentimen.

Tahap stemming tidak dilakukan dalam penelitian ini karena berpotensi menghilangkan konteks dan makna asli kata, yang penting dalam analisis sentimen, terutama saat menggunakan representasi seperti TF-IDF dan Word2Vec yang sensitif terhadap variasi bentuk kata.



# Install Library & Data Loading

In [ ]:
# INSTALL LIBRARY
!pip install pandas numpy nltk scikit-learn swifter emoji
!pip install sastrawi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.5 MB/s eta 0:00:00
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=2babfa1da5775d5b9f52e0771a00e9de3f3c96d9d61aabadf76ebf2df70b589a
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.2 MB/s eta 0:00:00


In [ ]:
# IMPORT LIBRARY
import pandas as pd
import numpy as np
import re
import emoji
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import swifter
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [ ]:
# LOAD DATASET
df = pd.read_csv("/content/ulasan_ovo_labeled_final.csv")

print("Dataset Info:")
print(df.info())
print("\nDataset Head:")
print(df.head())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   reviewId     20000 non-null  object
 1   content      20000 non-null  object
 2   label_final  20000 non-null  object
dtypes: object(3)
memory usage: 468.9+ KB
None

Dataset Head:
                               reviewId  \
0  a4e17d51-263a-4fe6-a400-1e4e97ab3e9f   
1  572d520d-c61c-47cf-80b7-47293b9761dd   
2  138520c3-6f8d-41c5-a11d-1a2f234a10f6   
3  28d45f83-7667-436a-8947-4b19d275be35   
4  590d340a-a226-4cc2-b98c-0d81e0d24322   

                                             content label_final  
0  Tolong saldo driver saya selesaikan prosesnya,...     Negatif  
1  sering pembaruan gak malah bagus malah sering ...     Negatif  
2  parah ini ovo, Top up saldo grab dari tadi mas...     Negatif  
3  sering gangguan yg menyebabkan kerugian bagi k...     Negatif  
4  Ini aplikasi g

# Dictionary

In [ ]:
#  KAMUS SLANG BAHASA INDONESIA (Play Store Reviews)
#  Mencakup: Slang umum, singkatan informal, bahasa daerah
#  (Sunda, Jawa, Betawi, Batak, Melayu, dll.)

slang_dict = {

    # A. SLANG UMUM & SINGKATAN INFORMAL
    "@": "di",
    "abis": "habis",
    "ad": "ada",
    "adlh": "adalah",
    "adlah": "adalah",
    "afaik": "as far as i know",
    "ahaha": "haha",
    "aj": "saja",
    "aja": "saja",
    "ajj": "saja",
    "ak": "saya",
    "akika": "aku",
    "akkoh": "aku",
    "akku": "aku",
    "akuwh": "aku",
    "akyu": "aku",
    "akko": "aku",
    "alay": "norak",
    "alow": "halo",
    "ambilin": "ambilkan",
    "ancur": "hancur",
    "anjrit": "anjing",
    "anjir": "anjing",
    "anter": "antar",
    "ap2": "apa-apa",
    "apasih": "apa sih",
    "apaan": "apa",
    "apes": "sial",
    "aps": "apa",
    "aq": "saya",
    "aquwh": "aku",
    "asbun": "asal bunyi",
    "aseekk": "asyik",
    "asekk": "asyik",
    "asem": "asam",
    "ato": "atau",
    "awak": "saya",
    "ay": "sayang",
    "ayank": "sayang",
    "b4": "sebelum",
    "bakalan": "akan",
    "bangedh": "banget",
    "basbang": "basi",
    "bcanda": "bercanda",
    "begajulan": "nakal",
    "beliin": "belikan",
    "bentar": "sebentar",
    "beresin": "membereskan",
    "bete": "bosan",
    "beud": "banget",
    "bg": "abang",
    "bgmn": "bagaimana",
    "bgt": "banget",
    "bngt": "banget",
    "bijimane": "bagaimana",
    "bkl": "akan",
    "bknnya": "bukannya",
    "blegug": "bodoh",
    "blh": "boleh",
    "bln": "bulan",
    "blum": "belum",
    "blm": "belum",
    "bnci": "benci",
    "bnran": "yang benar",
    "bodor": "lucu",
    "bokap": "ayah",
    "boker": "buang air besar",
    "bokis": "bohong",
    "boljug": "boleh juga",
    "boyeh": "boleh",
    "br": "baru",
    "brg": "bareng",
    "bro": "saudara laki-laki",
    "bru": "baru",
    "bs": "bisa",
    "bsa": "bisa",
    "bsen": "bosan",
    "bt": "buat",
    "btw": "ngomong-ngomong",
    "bw": "bawa",
    "bwt": "buat",
    "byk": "banyak",
    "bnyk": "banyak",
    "byrin": "bayarkan",
    "byar": "bayar",
    "byr": "bayar",
    "cabal": "sabar",
    "cadas": "keren",
    "caper": "cari perhatian",
    "ce": "cewek",
    "cemen": "penakut",
    "cepet": "cepat",
    "cew": "cewek",
    "ckp": "cakep",
    "cmiiw": "correct me if i'm wrong",
    "cmpur": "campur",
    "cucok": "cocok",
    "cuex": "cuek",
    "cups": "culun",
    "curcol": "curahan hati colongan",
    "cwek": "cewek",
    "d": "di",
    "dah": "deh",
    "dapet": "dapat",
    "de": "adik",
    "dek": "adik",
    "demen": "suka",
    "deyh": "deh",
    "dgn": "dengan",
    "dingn": "dengan",
    "diancurin": "dihancurkan",
    "dket": "dekat",
    "dll": "dan lain-lain",
    "dlu": "dulu",
    "dngn": "dengan",
    "dodol": "bodoh",
    "doku": "uang",
    "dpt": "dapat",
    "dri": "dari",
    "drmn": "darimana",
    "drtd": "dari tadi",
    "dst": "dan seterusnya",
    "dtg": "datang",
    "duh": "aduh",
    "egp": "emang gue pikirin",
    "eke": "aku",
    "elu": "kamu",
    "emangnya": "memangnya",
    "emang": "memang",
    "emng": "memang",
    "endak": "tidak",
    "enggak": "tidak",
    "ex": "mantan",
    "folbek": "follow back",
    "fyi": "sebagai informasi",
    "gaada": "tidak ada",
    "gag": "tidak",
    "gaje": "tidak jelas",
    "gajelas": "tidak jelas",
    "gak papa": "tidak apa-apa",
    "gaptek": "gagap teknologi",
    "gawe": "kerja",
    "gbs": "tidak bisa",
    "gabisa": "tidak bisa",
    "gebetan": "orang yang disuka",
    "geje": "tidak jelas",
    "ghiy": "lagi",
    "gile": "gila",
    "gimana": "bagaimana",
    "gmana": "bagaimana",
    "githu": "gitu",
    "gitu": "begitu",
    "gini": "begini",
    "gj": "tidak jelas",
    "gk": "tidak",
    "goblok": "bodoh",
    "gokil": "gila",
    "gombal": "suka merayu",
    "gpp": "tidak apa-apa",
    "gt": "begitu",
    "gtau": "tidak tahu",
    "gtw": "tidak tahu",
    "gua": "saya",
    "gue": "saya",
    "guys": "teman-teman",
    "gws": "cepat sembuh",
    "gw": "saya",
    "ha": "tertawa",
    "haha": "tertawa",
    "hallow": "halo",
    "hehe": "tertawa kecil",
    "helo": "halo",
    "hey": "hai",
    "hlm": "halaman",
    "hny": "hanya",
    "hoax": "isu bohong",
    "hr": "hari",
    "hrus": "harus",
    "huff": "mengeluh",
    "huft": "mengeluh",
    "ilang": "hilang",
    "ilfil": "tidak suka",
    "ilfeel": "tidak suka",
    "imho": "in my humble opinion",
    "item": "hitam",
    "itungan": "hitungan",
    "iye": "iya",
    "ja": "saja",
    "jadiin": "jadi",
    "jaim": "jaga image",
    "jayus": "tidak lucu",
    "jdi": "jadi",
    "jem": "jam",
    "jga": "juga",
    "jg": "juga",
    "jgn": "jangan",
    "jir": "anjing",
    "jln": "jalan",
    "jomblo": "tidak punya pacar",
    "jubir": "juru bicara",
    "jutek": "galak",
    "k": "ke",
    "kabor": "kabur",
    "kacrut": "kacau",
    "kagak": "tidak",
    "kaga": "tidak",
    "kalo": "kalau",
    "kampret": "sialan",
    "kamuwh": "kamu",
    "karna": "karena",
    "krn": "karena",
    "kayanya": "kayaknya",
    "kbr": "kabar",
    "kdu": "harus",
    "kekeuh": "keras kepala",
    "kemaren": "kemarin",
    "kepengen": "mau",
    "kepingin": "mau",
    "ketrima": "diterima",
    "kgiatan": "kegiatan",
    "kl": "kalau",
    "klo": "kalau",
    "klw": "kalau",
    "km": "kamu",
    "kmrn": "kemarin",
    "knal": "kenal",
    "knp": "kenapa",
    "kpn": "kapan",
    "krenz": "keren",
    "krm": "kirim",
    "kt": "kita",
    "ktmu": "ketemu",
    "ktr": "kantor",
    "kuper": "kurang pergaulan",
    "kw": "imitasi",
    "kyk": "seperti",
    "ky": "seperti",
    "kykny": "kayaknya",
    "la": "lah",
    "lam": "salam",
    "lebay": "berlebihan",
    "leh": "boleh",
    "lelet": "lambat",
    "lemot": "lambat",
    "lgi": "lagi",
    "lg": "lagi",
    "lgsg": "langsung",
    "liat": "lihat",
    "lht": "lihat",
    "lmyn": "lumayan",
    "lmyan": "lumayan",
    "lo": "kamu",
    "loe": "kamu",
    "lola": "lambat berfikir",
    "lom": "belum",
    "lum": "belum",
    "lwn": "lawan",
    "lwt": "lewat",
    "males": "malas",
    "mls": "malas",
    "macem": "macam",
    "maem": "makan",
    "mak jang": "kaget",
    "maksain": "memaksa",
    "malem": "malam",
    "mam": "makan",
    "maniez": "manis",
    "mao": "mau",
    "masukin": "masukkan",
    "mgu": "minggu",
    "mending": "lebih baik",
    "mgkn": "mungkin",
    "mhn": "mohon",
    "mksd": "maksud",
    "mlah": "malah",
    "mngkn": "mungkin",
    "mo": "mau",
    "moso": "masa",
    "mosok": "masa",
    "mpe": "sampai",
    "ampe": "sampai",
    "sampe": "sampai",
    "msk": "masuk",
    "mslh": "masalah",
    "msh": "masih",
    "musti": "mesti",
    "mw": "mau",
    "n": "dan",
    "nanya": "bertanya",
    "napa": "kenapa",
    "nda": "tidak",
    "ndak": "tidak",
    "ndiri": "sendiri",
    "ne": "ini",
    "nembak": "menyatakan cinta",
    "ngaku": "mengaku",
    "ngambil": "mengambil",
    "nganggur": "tidak punya pekerjaan",
    "ngapain": "sedang apa",
    "ngapah": "kenapa",
    "ngaret": "terlambat",
    "ngasih": "memberikan",
    "ngebandel": "berbuat bandel",
    "ngegosip": "bergosip",
    "ngeklaim": "mengklaim",
    "ngeles": "berkilah",
    "ngga": "tidak",
    "nggak": "tidak",
    "ngibul": "berbohong",
    "ngiler": "mau",
    "ngiri": "iri",
    "ngmng": "bicara",
    "ngomong": "bicara",
    "ngerti": "mengerti",
    "ngikut": "ikut",
    "ngisi": "mengisi",
    "ngomongin": "membicarakan",
    "ngumpul": "berkumpul",
    "ni": "ini",
    "nih": "ini",
    "niyh": "nih",
    "nmr": "nomor",
    "nomer": "nomor",
    "nntn": "nonton",
    "nobar": "nonton bareng",
    "ntar": "nanti",
    "tar": "nanti",
    "ntn": "nonton",
    "nunggu": "menunggu",
    "nutupin": "menutupi",
    "nyari": "mencari",
    "nyariin": "mencari",
    "nyicil": "mencicil",
    "nyangkut": "tersangkut",
    "nyesel": "menyesal",
    "nyekar": "menyekar",
    "nyiapin": "mempersiapkan",
    "nyok": "ayo",
    "ogah": "tidak mau",
    "ol": "online",
    "ongkir": "ongkos kirim",
    "org2": "orang-orang",
    "ortu": "orang tua",
    "otw": "sedang di jalan",
    "pake": "pakai",
    "pd": "pada",
    "pede": "percaya diri",
    "perhatiin": "perhatikan",
    "pesenan": "pesanan",
    "plg": "paling",
    "pst": "pasti",
    "psti": "pasti",
    "prg": "pergi",
    "prnh": "pernah",
    "psen": "pesan",
    "psn": "pesan",
    "pw": "posisi nyaman",
    "rakor": "rapat koordinasi",
    "re": "balas",
    "bales": "balas",
    "rempong": "sulit",
    "ribet": "sulit",
    "rhs": "rahasia",
    "rmh": "rumah",
    "ru": "baru",
    "ruz": "terus",
    "saia": "saya",
    "salting": "salah tingkah",
    "samsek": "sama sekali",
    "sbb": "sebagai berikut",
    "sbh": "sebuah",
    "sbnrny": "sebenarnya",
    "scr": "secara",
    "sdgkn": "sedangkan",
    "sdh": "sudah",
    "sdkt": "sedikit",
    "dikit": "sedikit",
    "sempet": "sempat",
    "sgt": "sangat",
    "shg": "sehingga",
    "sih": "sih",
    "sikon": "situasi dan kondisi",
    "sj": "saja",
    "skalian": "sekalian",
    "skt": "sakit",
    "slesai": "selesai",
    "slsai": "selesai",
    "sll": "selalu",
    "slma": "selama",
    "smpt": "sempat",
    "smw": "semua",
    "sndiri": "sendiri",
    "songong": "sombong",
    "sory": "maaf",
    "sorry": "maaf",
    "sowry": "maaf",
    "spt": "seperti",
    "sprti": "seperti",
    "spy": "supaya",
    "stelah": "setelah",
    "stlh": "setelah",
    "sy": "saya",
    "sya": "saya",
    "syp": "siapa",
    "t4": "tempat",
    "tajir": "kaya",
    "tau": "tahu",
    "taw": "tahu",
    "tauk": "tahu",
    "tw": "tahu",
    "td": "tadi",
    "tdi": "tadi",
    "tdk": "tidak",
    "telat": "terlambat",
    "temen": "teman",
    "temen2": "teman-teman",
    "tman": "teman",
    "tmn2": "teman-teman",
    "tggu": "tunggu",
    "tgu": "tunggu",
    "tngu": "tunggu",
    "thankz": "terima kasih",
    "thanks": "terima kasih",
    "thx": "terima kasih",
    "tks": "terima kasih",
    "trims": "terima kasih",
    "maacih": "terima kasih",
    "makasih": "terima kasih",
    "thn": "tahun",
    "taun": "tahun",
    "tgl": "tanggal",
    "tlp": "telepon",
    "tlpn": "telepon",
    "telp": "telepon",
    "tll": "terlalu",
    "trlalu": "terlalu",
    "tmbah": "tambah",
    "tmbh": "tambah",
    "tmpt": "tempat",
    "tnyta": "ternyata",
    "tp": "tapi",
    "tpi": "tapi",
    "tq": "terima kasih",
    "trs": "terus",
    "trus": "terus",
    "trm": "terima",
    "ttg": "tentang",
    "tuch": "tuh",
    "tuh": "itu",
    "u": "kamu",
    "ud": "sudah",
    "udah": "sudah",
    "udh": "sudah",
    "ul": "ulangan",
    "unyu": "lucu",
    "uplot": "unggah",
    "utk": "untuk",
    "wkt": "waktu",
    "wtf": "what the fuck",
    "xixixi": "tertawa",
    "ya": "iya",
    "yap": "iya",
    "yup": "iya",
    "iyo": "iya",
    "yaudah": "ya sudah",
    "yawdah": "ya sudah",
    "yowes": "ya sudah",
    "yg": "yang",
    "yl": "yang lain",
    "yo": "iya",
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "hadeh": "mengeluh",
    "hadehh": "mengeluh",
    "haddeh": "mengeluh",

    # -------------------------------------------------------
    # B. SLANG BARU DARI DATA ULASAN OVO (Play Store)
    # -------------------------------------------------------

    # Singkatan konteks e-wallet/aplikasi
    "apk": "aplikasi",
    "tf": "transfer",
    "topup": "isi ulang",
    "top up": "isi ulang",
    "gabisa": "tidak bisa",
    "eror": "error",
    "nomer": "nomor",
    "kepotong": "terpotong",
    "blokir": "diblokir",
    "nunggu": "menunggu",
    "cs": "customer service",
    "hp": "handphone",
    "qris": "qr payment",
    "e-wallet": "dompet digital",
    "ewallet": "dompet digital",
    "wallet": "dompet digital",
    "duit": "uang",
    "mantul": "mantap betul",
    "mantep": "mantap",

    # Ekspresi negatif umum di ulasan
    "parah": "sangat buruk",
    "mending": "lebih baik",
    "ribet": "susah",
    "lelet": "lambat",
    "lemot": "lambat",
    "payah": "buruk",
    "jelek": "buruk",
    "sampah": "tidak berguna",
    "bokbrok": "rusak parah",
    "ngotak": "tidak masuk akal",
    "gajelas": "tidak jelas",
    "bikin": "membuat",
    "bener": "benar",
    "kecewa": "kecewa",
    "nyesel": "menyesal",
    "nyangkut": "tertahan",
    "males": "malas",
    "mulu": "terus-terusan",
    "ampe": "sampai",
    "emang": "memang",
    "kaga": "tidak",
    "kagak": "tidak",
    "pol": "sangat / habis",
    "ruwet": "rumit",
    "kentang": "tidak kompeten",
    "pekok": "bodoh",
    "geblek": "bodoh",

    # Ekspresi tawa/gumaman
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "wkwkwkwk": "tertawa",
    "hehe": "tertawa kecil",
    "hihi": "tertawa kecil",
    "huhu": "menangis",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "xixi": "tertawa",

    # -------------------------------------------------------
    # C. BAHASA SUNDA (Slang & Informal)
    # -------------------------------------------------------
    "tong": "jangan",
    "ngaganggu": "mengganggu",
    "kagok": "canggung",
    "lalajo": "menonton",
    "maneh": "kamu",
    "aing": "saya",
    "urang": "saya",
    "abdi": "saya",
    "atuh": "dong",
    "euy": "ya",
    "mah": "sih",
    "bray": "bro",
    "kumaha": "bagaimana",
    "naon": "apa",
    "tos": "sudah",
    "teu": "tidak",
    "oge": "juga",
    "cenah": "katanya",
    "ceuk": "kata",
    "ngan": "hanya",
    "geuning": "ternyata",
    "pisan": "sangat",
    "ngarti": "mengerti",
    "embung": "tidak mau",
    "hayang": "mau",
    "teu ngartos": "tidak mengerti",
    "pa": "bapak",
    "bu": "ibu",
    "a": "kakak laki-laki",
    "teteh": "kakak perempuan",
    "aa": "kakak laki-laki",
    "punten": "permisi",
    "hatur nuhun": "terima kasih",
    "nuhun": "terima kasih",
    "sawios": "tidak apa-apa",
    "sieun": "takut",
    "gering": "sakit",
    "bener oge": "benar juga",
    "lain": "bukan",
    "pan": "kan",
    "ari": "kalau",
    "wae": "saja",
    "keneh": "masih",
    "mani": "sangat",
    "ngaranna": "namanya",
    "cio": "lihat",
    "ker": "lagi",
    "geus": "sudah",
    "can": "belum",
    "henteu": "tidak",
    "nempo": "melihat",
    "datang": "datang",
    "balik": "pulang",
    "leumpang": "jalan",
    "lumpat": "lari",
    "nyaho": "tahu",
    "micara": "berbicara",
    "ngomong": "berbicara",

    # -------------------------------------------------------
    # D. BAHASA JAWA (Slang & Informal)
    # -------------------------------------------------------
    "ora": "tidak",
    "opo": "apa",
    "ngopo": "ngapain",
    "piye": "bagaimana",
    "karepmu": "terserah kamu",
    "wes": "sudah",
    "wis": "sudah",
    "ae": "saja",
    "sek": "sebentar",
    "mbok": "kalau bisa",
    "lho": "loh",
    "ta": "kan",
    "to": "kan",
    "rek": "bro",
    "cak": "bro",
    "nin": "ini",
    "kae": "itu",
    "ngono": "begitu",
    "ngene": "begini",
    "iki": "ini",
    "kuwi": "itu",
    "kono": "sana",
    "kene": "sini",
    "loro": "dua",
    "telu": "tiga",
    "opo meneh": "apalagi",
    "yo wis": "ya sudah",
    "ndes": "goblok",
    "edan": "gila",
    "cok": "umpatan",
    "jancok": "umpatan kasar",
    "jancuk": "umpatan kasar",
    "lonthe": "umpatan",
    "pekok": "bodoh",
    "goblok": "bodoh",
    "mbuh": "tidak tahu",
    "ojo": "jangan",
    "nek": "kalau",
    "ndak": "tidak",
    "ra": "tidak",
    "gak usah": "tidak perlu",
    "jan": "sungguh",
    "jan parah": "sungguh parah",
    "jan edan": "sungguh gila",
    "yo ben": "ya sudah biarkan",
    "ngerti": "mengerti",
    "ngerti ora": "mengerti tidak",
    "isoh": "bisa",
    "gelem": "mau",
    "ruwet": "rumit",
    "uwes": "sudah",
    "mas": "kakak laki-laki",
    "mbak": "kakak perempuan",
    "mbah": "nenek/kakek",
    "le": "sapaan anak laki-laki",
    "neng": "sapaan anak perempuan",
    "pak": "bapak",
    "bu": "ibu",
    "raimu": "mukamu",
    "jian": "sungguh",
    "sembarangan": "asal-asalan",

    # -------------------------------------------------------
    # E. BAHASA BETAWI
    # -------------------------------------------------------
    "aye": "saya",
    "ente": "kamu",
    "agan": "kakak",
    "gan": "juragan",
    "nyak": "ibu",
    "babe": "ayah",
    "kagak": "tidak",
    "kaga": "tidak",
    "iye": "iya",
    "die": "dia",
    "gue": "saya",
    "lo": "kamu",
    "lu": "kamu",
    "nape": "kenapa",
    "apaan": "apa",
    "beneran": "sungguh",
    "mane": "mana",
    "ame": "dengan",
    "ame-ame": "sama-sama",
    "sono": "sana",
    "sini": "sini",
    "nyang": "yang",
    "nyari": "mencari",
    "makan mulu": "makan terus",
    "bokap": "ayah",
    "nyokap": "ibu",
    "bonyok": "orang tua",
    "duit": "uang",
    "kite": "kita",
    "nih": "ini",
    "tuh": "itu",

    # -------------------------------------------------------
    # F. BAHASA MELAYU INFORMAL
    # -------------------------------------------------------
    "lah": "lah",
    "leh": "boleh",
    "tak": "tidak",
    "nak": "mau",
    "dah": "sudah",
    "pun": "juga",
    "mana ada": "tidak ada",
    "camne": "bagaimana",
    "camana": "bagaimana",
    "napa": "kenapa",
    "tanak": "tidak mau",
    "apa hal": "ada apa",
    "kan": "kan",
    "pe": "apa",
    "kinek": "sekarang",
    "sik": "saja",
    "kelak": "nanti",
    "agik": "lagi",
    "juak": "juga",
    "kitak": "kalian",
    "aku": "saya",
    "dolok": "dulu",
    "sitok": "satu",
    "gawai": "pekerjaan",
    "endak": "tidak",

    # -------------------------------------------------------
    # G. BAHASA BATAK (Informal)
    # -------------------------------------------------------
    "horas": "salam",
    "amang": "ayah",
    "inang": "ibu",
    "lae": "kakak",
    "ito": "saudara",
    "dongan": "teman",
    "manjolo": "duluan",
    "bah": "bah",
    "jolo": "dulu",
    "buha": "buka",
    "ndang": "tidak",
    "nda": "tidak",
    "nga": "tidak",
    "da": "sudah",
    "pe": "juga",
    "molo": "kalau",
    "di": "di",
    "nauli": "bagus",
    "godang": "besar",
    "denggan": "baik",

    # -------------------------------------------------------
    # H. BAHASA MINANG (Informal)
    # -------------------------------------------------------
    "ambo": "saya",
    "waang": "kamu",
    "awak": "saya",
    "den": "saya",
    "uda": "kakak laki-laki",
    "uni": "kakak perempuan",
    "etek": "tante",
    "mamak": "paman",
    "ka": "ke",
    "lai": "lagi",
    "iko": "ini",
    "tu": "itu",
    "nio": "mau",
    "indak": "tidak",
    "ndak": "tidak",
    "manga": "kenapa",
    "bara": "berapa",
    "bilo": "kapan",
    "kama": "kemana",
    "dimaa": "dimana",
    "baitu": "begitu",
    "baiko": "begini",
    "alah": "sudah",
    "lah": "sudah",
    "picayo": "percaya",
    "ndk": "tidak",
    "lamo": "lama",
    "rancak": "bagus",
    "elok": "baik",
    "manga ko": "kenapa ini",

    # -------------------------------------------------------
    # I. BAHASA MAKASSAR / BUGIS (Informal)
    # -------------------------------------------------------
    "ki": "anda",
    "ji": "saja",
    "mi": "sudah",
    "pi": "dulu",
    "ko": "kamu",
    "ku": "saya",
    "na": "dia",
    "ta": "kita",
    "di": "di",
    "iye": "iya",
    "tena": "tidak ada",
    "niak": "ada",
    "engka": "ada",
    "de": "tidak",
    "iyo": "iya",
    "aga": "apa",
    "pura": "sudah",
    "narekko": "kalau",
    "iye to": "iya dong",
    "begitu ji": "begitu saja",
    "pale": "kalau begitu",

    # -------------------------------------------------------
    # J. BAHASA AMBON / MALUKU (Informal)
    # -------------------------------------------------------
    "beta": "saya",
    "katong": "kita",
    "dong": "mereka",
    "ose": "kamu",
    "ale": "kamu",
    "su": "sudah",
    "tra": "tidak",
    "bisa tra": "bisa tidak",
    "mar": "tapi",
    "deng": "dengan",
    "sonde": "tidak",
    "bagitu": "begitu",
    "mau apa": "mau apa",
    "pung": "punya",
    "bikin": "membuat",

    # -------------------------------------------------------
    # K. BAHASA PAPUA (Informal)
    # -------------------------------------------------------
    "kitorang": "kita",
    "dorang": "mereka",
    "ko": "kamu",
    "su": "sudah",
    "tra": "tidak",
    "kam": "kalian",
    "abis": "habis",
    "pi": "pergi",
    "kam pu": "punya kalian",

    # -------------------------------------------------------
    # L. SINGKATAN KHUSUS KONTEKS APLIKASI / DIGITAL
    # -------------------------------------------------------
    "apk": "aplikasi",
    "app": "aplikasi",
    "apl": "aplikasi",
    "apknya": "aplikasinya",
    "tf": "transfer",
    "tranfer": "transfer",
    "topup": "isi ulang",
    "gabisa": "tidak bisa",
    "gabisa masuk": "tidak bisa masuk",
    "gak bisa masuk": "tidak bisa login",
    "eror": "error",
    "error": "error",
    "nomer": "nomor",
    "no hp": "nomor handphone",
    "kepotong": "terpotong",
    "blokir": "diblokir",
    "diblokir": "diblokir",
    "nunggu": "menunggu",
    "otp": "one time password",
    "cs": "customer service",
    "hp": "handphone",
    "qris": "pembayaran QR",
    "e-wallet": "dompet digital",
    "ewallet": "dompet digital",
    "wallet": "dompet",
    "saldo": "saldo",
    "topup": "isi ulang",
    "log in": "masuk akun",
    "login": "masuk akun",
    "log out": "keluar akun",
    "logout": "keluar akun",
    "upgrade": "tingkatkan",
    "premier": "premium",
    "premium": "premium",
    "ktp": "kartu tanda penduduk",
    "pin": "personal identification number",
    "pending": "tertunda",
    "proses": "sedang diproses",
    "laporan": "laporan",
    "developer": "pengembang",
    "devoloper": "pengembang",
    "install": "pasang",
    "uninstall": "hapus",
    "update": "perbarui",
    "download": "unduh",
    "upload": "unggah",
    "screenshot": "tangkapan layar",
    "ss": "screenshot",
    "notif": "notifikasi",
    "server": "peladen",
    "maintenance": "pemeliharaan",
    "down": "tidak bisa diakses",
    "laload": "lambat loading",
    "loading": "memuat",
    "lag": "lambat",
    "crash": "mogok",
    "bug": "kesalahan sistem",
    "fitur": "fitur",
    "rekening": "rekening",
    "mutasi": "riwayat transaksi",
    "admin": "biaya administrasi",
    "potongan": "potongan biaya",
    "gratis": "gratis",
    "promo": "promosi",
    "cashback": "uang kembali",
    "voucher": "voucher",

    # -------------------------------------------------------
    # M. TAMBAHAN SLANG GAUL KONTEMPORER
    # -------------------------------------------------------
    "santuy": "santai",
    "gercep": "gerak cepat",
    "kepo": "mau tahu",
    "baper": "bawa perasaan",
    "mager": "malas gerak",
    "gabut": "tidak ada kerjaan",
    "japri": "jalur pribadi",
    "dm": "pesan langsung",
    "stalker": "pengintai",
    "ghosting": "menghilang",
    "php": "pemberi harapan palsu",
    "bales": "membalas",
    "respon": "merespons",
    "chat": "pesan",
    "wa": "whatsapp",
    "ig": "instagram",
    "tiktok": "tiktok",
    "sosmed": "media sosial",
    "konten": "konten",
    "feed": "beranda",
    "viral": "viral",
    "trending": "sedang tren",
    "hits": "populer",
    "kekinian": "terkini",
    "jadul": "jaman dulu",
    "ngehits": "populer",
    "sultan": "orang kaya",
    "sultan mode": "mode orang kaya",
    "receh": "sepele",
    "zonk": "kosong",
    "mantul": "mantap betul",
    "cuan": "keuntungan",
    "bokek": "tidak punya uang",
    "kere": "tidak punya uang",
    "tajir": "kaya",
    "melintir": "gila",
    "aduhai": "luar biasa",
    "anjay": "ekspresi kaget",
    "wkwkwk": "tertawa",
    "lmao": "tertawa keras",
    "omg": "oh my god",
    "asap": "sesegera mungkin",
    "dll": "dan lain-lain",
    "dsb": "dan sebagainya",
    "dst": "dan seterusnya",
    "etc": "dan lain-lain",
    "ok": "oke",
    "oke": "oke",
    "sip": "oke",
    "iyap": "iya",
    "nope": "tidak",
    "yep": "ya",
}

print(f"Total entri kamus slang: {len(slang_dict)}")

Total entri kamus slang: 839


In [ ]:
# KATA NEGASI YANG HARUS DIPERTAHANKAN
NEGATION_WORDS = {
    'tidak', 'gak', 'ga', 'gk', 'ngga', 'nggak', 'tak', 'enggak',
    'bukan', 'bkn', 'belum', 'blm', 'blum', 'jangan', 'jgn',
    'kurang', 'tanpa', 'bukanlah', 'takkan', 'janganlah',
    'ora', 'teu', 'tra', 'sonde', 'indak', 'ndak', 'nda',
    'never', 'no', 'not', 'none', 'nothing', 'nobody'
}

In [ ]:

# STOPWORDS YANG AMAN (TIDAK TERMASUK NEGASI)
SAFE_STOPWORDS = {
    # Partikel
    "sih", "nih", "ni", "tuh", "deh", "lah", "dong", "dah",
    "nah", "yah", "yaa", "yaah", "wah", "woi", "woii", "woy",
    "eh", "aduh", "ah", "ih", "oh", "uh", "hm", "hmm",
    "hadeh", "hadehh", "haddeh",
    "hehe", "haha", "hihi", "huhu",
    "wkwk", "wkwkwk", "wkwkwkwk",
    "xixi", "xixixi",

    # Kata ganti
    "gw", "gue", "gua", "sy", "sya", "aq", "ak", "ku",
    "beta", "awak", "den", "ambo", "urang", "aing", "aye",
    "lo", "lu", "u", "km", "ente", "maneh", "waang", "ko",
    "nya", "ny", "doi", "katong", "kitorang",

    # Singkatan umum
    "yg", "dgn", "dngn", "dng", "utk", "dr", "dri", "pd",
    "dlm", "sdh", "udah", "udh", "bgt", "bngt", "jg", "jd",
    "tp", "tpi", "klo", "kl", "klw", "kalo", "trs", "trus",
    "lg", "lgi", "knp", "knpa", "gmn", "gmna", "gimna",
    "skrg", "pdhl", "karna", "krn",

    # Huruf tunggal
    "a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k",
    "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v",
    "w", "x", "y", "z", "rp", "rb", "jt",

    # Filler
    "aja", "aj", "saja", "doang", "cuma", "cuman", "pun",
    "kok", "kan", "lho", "loh", "wes", "wis", "ya", "yap",
    "yup", "iya", "iyah", "iyo", "iye", "ok", "oke", "sip",

    # Sapaan
    "min", "kak", "pak", "bu", "mas", "mbak", "bang", "bg",
    "bro", "guys", "gan",

    # Keterangan waktu
    "tadi", "td", "tgl", "kemaren", "kmrn", "ntar", "tar",
    "kapan", "kpn", "dulu", "dlu",

    # Kata kerja bantu
    "adalah", "ialah", "yaitu", "yakni", "bagi", "buat",
    "pada", "dalam", "atas", "bawah", "samping", "antara",

    # Bahasa daerah (function words)
    "mah", "atuh", "euy", "teh", "wae", "pan", "ari", "geus", "can",
    "ae", "yo", "ta", "to", "je", "rek", "cak", "nak",
    "pe", "sik", "agik", "jua", "ji", "mi", "pi", "ki", "na",
    "su", "mar", "deng", "bah", "da",

    # Angka
    "satu", "dua", "tiga", "empat", "lima", "enam", "tujuh",
    "delapan", "sembilan", "sepuluh", "ribu", "juta", "persen",
    "jam", "menit", "detik", "hari", "bulan", "tahun", "minggu",

    # Aplikasi
    "ovo","aplikasi","apk",
}


# Data Cleaning Function

## Case Folding

In [ ]:
# FUNCTION: CASE FOLDING
def case_folding(text):
    """Mengubah teks menjadi lowercase"""
    if not isinstance(text, str):
        return text
    return text.lower()

## Normalize Numbers

In [ ]:
# FUNCTION: NORMALIZE NUMBERS
def normalize_numbers(text):
    """
    Normalisasi angka dan nominal uang menjadi token khusus
    Agar model bisa belajar pola tanpa overfit ke angka spesifik
    """
    if not isinstance(text, str):
        return text

    # Nominal Rupiah: Rp50.000, Rp 50.000, Rp50rb, dll
    text = re.sub(r'Rp\s*\d+(?:[.,]\d{3})*(?:[.,]\d+)?', '[harga]', text)
    text = re.sub(r'Rp\s*\d+\s*(?:rb|ribu)', '[harga]', text)
    text = re.sub(r'Rp\s*\d+\s*(?:jt|juta)', '[harga]', text)

    # Persentase
    text = re.sub(r'\d+(?:[.,]\d+)?%', '[persen]', text)
    text = re.sub(r'\d+\s*(?:persen|%)', '[persen]', text)

    # Angka biasa (opsional, bisa dihapus atau dipertahankan)
    # text = re.sub(r'\b\d+(?:[.,]\d+)?\b', '[angka]', text)

    return text

## Emoji's Handling

In [ ]:
# FUNCTION: HANDLE EMOJI KE SENTIMEN
def handle_emoji_sentiment(text):
    """
    Mapping emoji ke label sentimen atau kata deskriptif
    """
    if not isinstance(text, str):
        return text

    # Emoji positif
    positive_emoji = {
        '👍': 'positif', '👌': 'positif', '✅': 'positif',
        '❤️': 'suka', '💖': 'suka', '💕': 'suka',
        '😊': 'senang', '😄': 'senang', '😁': 'senang',
        '😂': 'tertawa', '🤣': 'tertawa', '🥰': 'sayang',
        '🎉': 'raya', '🔥': 'keren', '⭐': 'bagus',
        '🏆': 'juara', '💯': 'sempurna'
    }

    # Emoji negatif
    negative_emoji = {
        '👎': 'negatif', '💔': 'kecewa', '😡': 'marah',
        '😠': 'marah', '🤬': 'marah', '😤': 'kesal',
        '😭': 'sedih', '😢': 'sedih', '😥': 'kecewa',
        '💀': 'buruk', '👿': 'jahat', '😱': 'kaget',
        '🤮': 'muak', '💩': 'buruk'
    }

    for emoji_char, sentiment in positive_emoji.items():
        text = text.replace(emoji_char, f' {sentiment} ')

    for emoji_char, sentiment in negative_emoji.items():
        text = text.replace(emoji_char, f' {sentiment} ')

    # Sisa emoji diubah ke teks deskriptif
    text = emoji.demojize(text, delimiters=(" ", " "))

    return text

## Remove Special Characters

In [ ]:
# FUNCTION: REMOVE SPECIAL CHARACTERS
def remove_special_characters(text):
    """Menghapus karakter spesial, hanya huruf dan spasi"""
    if not isinstance(text, str):
        return text

    # Hanya huruf a-z dan spasi
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

## Normalize Slang Words

In [ ]:
# FUNCTION: NORMALIZE SLANG
def normalize_text(text, slang_dict):
    """Normalisasi slang ke kata baku"""
    if not isinstance(text, str):
        return text

    text = text.lower().strip()

    # Ganti multi-kata terlebih dahulu (urutan descending by length)
    for slang, formal in sorted(slang_dict.items(), key=lambda x: -len(x[0])):
        # Gunakan word boundary agar tidak replace substring
        pattern = r'\b' + re.escape(slang) + r'\b'
        text = re.sub(pattern, formal, text, flags=re.IGNORECASE)

    return text

## Tokenization

In [ ]:
# ============================================================
# FUNCTION: TOKENIZATION
# ============================================================
def tokenize_text(text):
    """Tokenisasi menggunakan NLTK word_tokenize"""
    if not isinstance(text, str):
        return []
    return word_tokenize(text)

## Remove Stopwords

In [ ]:
# FUNCTION: REMOVE STOPWORDS
def remove_stopwords_safe(tokens, keep_negations=True):
    """
    Menghapus stopwords tapi MEMPERTAHANKAN kata negasi
    """
    if not tokens:
        return []

    # Stopwords dari NLTK (Bahasa Indonesia)
    stop_words = set(stopwords.words('indonesian'))

    # Tambahkan safe stopwords
    stop_words.update(SAFE_STOPWORDS)

    # Jika keep_negations=True, hapus kata negasi dari stopwords
    if keep_negations:
        stop_words = stop_words - NEGATION_WORDS

    return [word for word in tokens if word not in stop_words]

## Negation Marking

In [ ]:
# FUNCTION: NEGATION MARKING
def mark_negations(tokens, neg_words=None):
    """
    Mark negation words with NOT_ prefix for better sentiment detection

    Example:
    ["saya", "tidak", "suka"] -> ["saya", "NOT_suka"]
    ["aplikasi", "kurang", "bagus"] -> ["aplikasi", "kurang_bagus"]
    """
    if not tokens:
        return []

    if neg_words is None:
        neg_words = NEGATION_WORDS

    result = []
    negate_next = False
    neg_phrase = []

    for i, token in enumerate(tokens):
        if token in neg_words:
            negate_next = True
            neg_phrase = [token]
        elif negate_next:
            # Gabungkan negasi dengan kata berikutnya
            result.append(f"NOT_{token}")
            negate_next = False
            neg_phrase = []
        else:
            result.append(token)

    return result

## Stemming

In [ ]:
# FUNCTION: STEMMING (OPSIONAL - SET FALSE UNTUK SENTIMEN)
def stemming(tokens, use_stemming=False):
    """
    Stemming menggunakan Sastrawi
    REKOMENDASI: SET use_stemming=False untuk sentiment analysis
    """
    if not tokens or not use_stemming:
        return tokens

    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    return [stemmer.stem(token) for token in tokens]

## Main PreProcessing Pipeline

In [ ]:
# FUNCTION: MAIN PREPROCESSING PIPELINE
def preprocess_for_sentiment(text, slang_dict,
                             use_emoji=True,
                             use_stemming=False,
                             use_negation_marking=True,
                             keep_negations=True):
    """
    PIPELINE UTAMA UNTUK SENTIMENT ANALYSIS

    Parameters:
    -----------
    text : str
        Teks raw dari dataset
    slang_dict : dict
        Kamus slang
    use_emoji : bool
        Apakah handling emoji (default True)
    use_stemming : bool
        Apakah stemming (default False - TIDAK DISARANKAN)
    use_negation_marking : bool
        Apakah marking negasi (default True - SANGAT DISARANKAN)
    keep_negations : bool
        Apakah mempertahankan kata negasi di stopwords (default True)

    Returns:
    --------
    tuple : (text_cleaned, tokens, tokens_with_negation, tokens_stemmed)
    """
    if not isinstance(text, str):
        return '', [], [], []

    # 1. Case folding
    text = case_folding(text)

    # 2. Normalisasi angka/harga
    text = normalize_numbers(text)

    # 3. Emoji handling (sentiment-aware)
    if use_emoji:
        text = handle_emoji_sentiment(text)

    # 4. Hapus karakter spesial
    text = remove_special_characters(text)

    # 5. Normalisasi slang
    text = normalize_text(text, slang_dict)

    # 6. Tokenization
    tokens = tokenize_text(text)

    # 7. Stopwords removal (safe - keep negations)
    tokens_cleaned = remove_stopwords_safe(tokens, keep_negations=keep_negations)

    # 8. Negation marking (IMPORTANT for sentiment!)
    if use_negation_marking:
        tokens_with_negation = mark_negations(tokens_cleaned)
    else:
        tokens_with_negation = tokens_cleaned

    # 9. Stemming (optional - default FALSE)
    if use_stemming:
        tokens_stemmed = stemming(tokens_with_negation, use_stemming=True)
    else:
        tokens_stemmed = tokens_with_negation

    # Convert to string
    text_cleaned = ' '.join(tokens_cleaned)
    text_with_negation = ' '.join(tokens_with_negation)
    text_stemmed = ' '.join(tokens_stemmed)

    return text_cleaned, tokens_cleaned, tokens_with_negation, tokens_stemmed

# Data Cleaning

In [ ]:
# PREPROCESS SELURUH DATASET
print("\n" + "="*80)
print("PREPROCESSING SELURUH DATASET")
print("="*80)
print(f"Jumlah data: {len(df)}")

# Apply preprocessing ke seluruh dataset
print("Memproses... (menggunakan swifter untuk akselerasi)")

# Proses dengan parameter optimal untuk sentimen
df['text_cleaned'], df['tokens'], df['tokens_with_negation'], df['tokens_stemmed'] = zip(*df['content'].swifter.apply(
    lambda x: preprocess_for_sentiment(
        x, slang_dict,
        use_emoji=True,
        use_stemming=False,  # KRITIS: FALSE untuk sentimen!
        use_negation_marking=True,
        keep_negations=True
    )
))

print("✅ Preprocessing selesai!")


PREPROCESSING SELURUH DATASET
Jumlah data: 20000
Memproses... (menggunakan swifter untuk akselerasi)


Pandas Apply:   0%|          | 0/20000 [00:00<?, ?it/s]

✅ Preprocessing selesai!


In [ ]:
# SAMPLE HASIL PREPROCESSING
print("\n" + "="*80)
print("SAMPLE HASIL PREPROCESSING")
print("="*80)

for i in range(min(5, len(df))):
    print(f"\n[{i+1}] Original : {df.iloc[i]['content'][:100]}...")
    print(f"    Cleaned  : {df.iloc[i]['text_cleaned'][:100]}...")
    print(f"    With Neg : {df.iloc[i]['tokens_with_negation'][:10]}...")
    print(f"    Label    : {df.iloc[i]['label_final']}")
    print("-"*60)


SAMPLE HASIL PREPROCESSING

[1] Original : Tolong saldo driver saya selesaikan prosesnya, udah nunggu 24 jam ini, katanya abis 24 jam saldo bak...
    Cleaned  : tolong saldo driver selesaikan prosesnya menunggu habis saldo dibalikin saldo gak kunjung pulang tra...
    With Neg : ['tolong', 'saldo', 'driver', 'selesaikan', 'prosesnya', 'menunggu', 'habis', 'saldo', 'dibalikin', 'saldo']...
    Label    : Negatif
------------------------------------------------------------

[2] Original : sering pembaruan gak malah bagus malah sering error,...
    Cleaned  : pembaruan gak bagus error...
    With Neg : ['pembaruan', 'NOT_bagus', 'error']...
    Label    : Negatif
------------------------------------------------------------

[3] Original : parah ini ovo, Top up saldo grab dari tadi masih proses mulu....
    Cleaned  : buruk isi ulang saldo grab diproses terus-terusan...
    With Neg : ['buruk', 'isi', 'ulang', 'saldo', 'grab', 'diproses', 'terus-terusan']...
    Label    : Negatif
------

In [ ]:
# HAPUS BARIS YANG MENJADI NaN ATAU KOSONG SETELAH PREPROCESSING
print("\n" + "="*80)
print("MEMERIKSA DAN MENGHAPUS DATA KOSONG/Nan SETELAH PREPROCESSING")
print("="*80)

# Cek jumlah baris sebelum pembersihan
jumlah_awal = len(df)
print(f"Jumlah data sebelum pembersihan: {jumlah_awal}")

# 1. Cek baris yang memiliki NaN di kolom penting
nan_check = df[['text_cleaned', 'tokens', 'tokens_with_negation']].isna().sum()
print(f"\nJumlah NaN per kolom:")
print(f"  - text_cleaned: {nan_check['text_cleaned']} baris")
print(f"  - tokens: {nan_check['tokens']} baris")
print(f"  - tokens_with_negation: {nan_check['tokens_with_negation']} baris")

# 2. Hapus baris yang memiliki NaN
df = df.dropna(subset=['text_cleaned', 'tokens', 'tokens_with_negation'])
print(f"\n✅ Setelah hapus NaN: {len(df)} baris (berkurang {jumlah_awal - len(df)})")

# 3. Cek baris yang text_cleaned-nya kosong (string kosong setelah strip)
jumlah_sebelum_hapus_kosong = len(df)
df = df[df['text_cleaned'].str.strip() != '']
df = df[df['tokens_with_negation'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)]
print(f"✅ Setelah hapus string kosong: {len(df)} baris (berkurang {jumlah_sebelum_hapus_kosong - len(df)})")

# 4. Cek baris yang hanya berisi stopwords semua (tidak ada kata bermakna)
def has_meaningful_words(tokens):
    """Cek apakah ada token yang tidak berisi NOT_ dan panjang > 1"""
    if not isinstance(tokens, list):
        return False
    # Filter token yang bermakna (bukan NOT_ prefix saja)
    meaningful = [t for t in tokens if not t.startswith('NOT_') and len(t) > 1]
    return len(meaningful) > 0

jumlah_sebelum_filter = len(df)
df = df[df['tokens_with_negation'].apply(has_meaningful_words)]
print(f"✅ Setelah hapus tanpa kata bermakna: {len(df)} baris (berkurang {jumlah_sebelum_filter - len(df)})")

# 5. Reset index setelah pembersihan
df = df.reset_index(drop=True)

print(f"\n{'='*50}")
print(f"📊 RINGKASAN PEMBERSIHAN:")
print(f"   Data awal                : {jumlah_awal}")
print(f"   Data akhir               : {len(df)}")
print(f"   Data terbuang            : {jumlah_awal - len(df)} ({((jumlah_awal - len(df))/jumlah_awal)*100:.2f}%)")
print(f"{'='*50}")

# 6. Tampilkan sample data yang tersisa
print(f"\n✅ Sample data setelah pembersihan:")
for i in range(min(3, len(df))):
    print(f"  [{i}] {df.iloc[i]['content'][:80]}...")
    print(f"      Tokens: {df.iloc[i]['tokens_with_negation'][:8]}...")


MEMERIKSA DAN MENGHAPUS DATA KOSONG/Nan SETELAH PREPROCESSING
Jumlah data sebelum pembersihan: 20000

Jumlah NaN per kolom:
  - text_cleaned: 0 baris
  - tokens: 0 baris
  - tokens_with_negation: 0 baris

✅ Setelah hapus NaN: 20000 baris (berkurang 0)
✅ Setelah hapus string kosong: 19257 baris (berkurang 743)
✅ Setelah hapus tanpa kata bermakna: 18997 baris (berkurang 260)

📊 RINGKASAN PEMBERSIHAN:
   Data awal                : 20000
   Data akhir               : 18997
   Data terbuang            : 1003 (5.01%)

✅ Sample data setelah pembersihan:
  [0] Tolong saldo driver saya selesaikan prosesnya, udah nunggu 24 jam ini, katanya a...
      Tokens: ['tolong', 'saldo', 'driver', 'selesaikan', 'prosesnya', 'menunggu', 'habis', 'saldo']...
  [1] sering pembaruan gak malah bagus malah sering error,...
      Tokens: ['pembaruan', 'NOT_bagus', 'error']...
  [2] parah ini ovo, Top up saldo grab dari tadi masih proses mulu....
      Tokens: ['buruk', 'isi', 'ulang', 'saldo', 'grab', 'diproses

# Save Data

In [ ]:
# SIMPAN HASIL PREPROCESSING
df_final = df.copy()

# Pilih kolom yang akan disimpan
columns_to_save = [
    'reviewId', 'content', 'label_final',
    'text_cleaned',
    'tokens',
    'tokens_with_negation',  # ← UTAMA untuk sentiment analysis!
    'tokens_stemmed',
    'original_length',
    'cleaned_length',
    'token_count',
    'token_neg_count'
]

# Pastikan semua kolom ada
available_columns = [col for col in columns_to_save if col in df_final.columns]
df_final_to_save = df_final[available_columns]

# Save ke CSV
output_file = "/content/ulasan_ovo_preprocessed_optimized.csv"
df_final_to_save.to_csv(output_file, index=False)
print(f"\n✅ Data hasil preprocessing disimpan ke: {output_file}")


✅ Data hasil preprocessing disimpan ke: /content/ulasan_ovo_preprocessed_optimized.csv
